# Preprocessing for the baseline condition
I split these to have different models for different conditions.
Regime A — Baseline Operation:
- Nominal temperature and pressure
- Efficient cooling
- Lower fault probability
- Represents stable long-term operation

In [1]:
import pandas as pd

In [2]:
data=pd.read_csv('archive/chemical_process_timeseries.csv')

In [3]:
df = data[data['operating_regime']=='A'] #filtering for the condition

In [4]:
df['operating_regime'].value_counts()

operating_regime
A    388800
Name: count, dtype: int64

In [5]:
df.drop(columns='operating_regime',inplace=True)  #deleting the column

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 388800 entries, 0 to 388799
Data columns (total 20 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   timestamp             388800 non-null  str    
 1   reactor_id            388800 non-null  str    
 2   ambient_temp_effect   365600 non-null  float64
 3   reactor_temp          365285 non-null  float64
 4   reactor_pressure      365396 non-null  float64
 5   feed_flow_rate        365354 non-null  float64
 6   coolant_flow_rate     365436 non-null  float64
 7   agitator_speed_rpm    365316 non-null  float64
 8   reaction_rate         365284 non-null  float64
 9   conversion_rate       365656 non-null  float64
 10  selectivity           365085 non-null  float64
 11  yield_pct             365529 non-null  float64
 12  vibration_rms         365677 non-null  float64
 13  motor_current         365472 non-null  float64
 14  power_consumption_kw  365626 non-null  float64
 15  temp_setpoi

In [7]:
df['timestamp']= pd.to_datetime(df['timestamp'])
df['reactor_id'] = df['reactor_id'].str[-1:] #so I only have the reactornumber as id
df['reactor_id'] = df['reactor_id'].astype('int')

## Handling NaNs

In [8]:
#lets start at the top and work our way down
df['ambient_temp_effect'].sort_values() #it's the temperature effect, not the outside temperature itself, so it might be different for each of the 3 reactors. but since it only gradually changes (time-sensitive), forward/backwardfill makes sense here (but for each single reactor)
mask = df['reactor_id'] == 1 #for the first reactor
df.loc[mask, 'ambient_temp_effect'] = df.loc[mask, 'ambient_temp_effect'].ffill()

In [9]:
#reactor temperature - I would handle that similar
df.loc[mask, 'reactor_temp'] = df.loc[mask, 'reactor_temp'].interpolate(method='linear')
#but I changed ffill to interpolate: takes the mean from the next and the last entry

#reactor_pressure
df.loc[mask, 'reactor_pressure'] = df.loc[mask, 'reactor_pressure'].interpolate(method='linear')
#feed_flow_rate
df.loc[mask, 'feed_flow_rate'] = df.loc[mask, 'feed_flow_rate'].interpolate(method='linear')
#coolant_flow_rate
df.loc[mask, 'coolant_flow_rate'] = df.loc[mask, 'coolant_flow_rate'].interpolate(method='linear')
#agitator_speed_rpm
df.loc[mask, 'agitator_speed_rpm'] = df.loc[mask, 'agitator_speed_rpm'].interpolate(method='linear')
#reaction_rate
df.loc[mask, 'reaction_rate'] = df.loc[mask, 'reaction_rate'].interpolate(method='linear')
#conversion_rate
df.loc[mask, 'conversion_rate'] = df.loc[mask, 'conversion_rate'].interpolate(method='linear')
#selectivity
df.loc[mask, 'selectivity'] = df.loc[mask, 'selectivity'].interpolate(method='linear')
#yield_pct
df.loc[mask, 'yield_pct'] = df.loc[mask, 'yield_pct'].interpolate(method='linear')
#vibration_rms
df.loc[mask, 'vibration_rms'] = df.loc[mask, 'vibration_rms'].interpolate(method='linear')
#motor_current
df.loc[mask, 'motor_current'] = df.loc[mask, 'motor_current'].interpolate(method='linear')
#power_consumption_kw
df.loc[mask, 'power_consumption_kw'] = df.loc[mask, 'power_consumption_kw'].interpolate(method='linear')
#temp_setpoint
df['temp_setpoint']=df['temp_setpoint'].fillna(df['temp_setpoint'].median())#its all the same temp anyway
#pressure_setpoint
df['pressure_setpoint'] = df['pressure_setpoint'].fillna(df['pressure_setpoint'].median())#same here
#efficiency_loss_pct
df.loc[mask, 'efficiency_loss_pct'] = df.loc[mask, 'efficiency_loss_pct'].interpolate(method='linear')

In [10]:
df.columns

Index(['timestamp', 'reactor_id', 'ambient_temp_effect', 'reactor_temp',
       'reactor_pressure', 'feed_flow_rate', 'coolant_flow_rate',
       'agitator_speed_rpm', 'reaction_rate', 'conversion_rate', 'selectivity',
       'yield_pct', 'vibration_rms', 'motor_current', 'power_consumption_kw',
       'temp_setpoint', 'pressure_setpoint', 'fault_type',
       'efficiency_loss_pct', 'time_to_fault_min'],
      dtype='str')

In [11]:
#same for the other reactors but a little more handy:

columns = ['ambient_temp_effect', 'reactor_temp',
       'reactor_pressure', 'feed_flow_rate', 'coolant_flow_rate',
       'agitator_speed_rpm', 'reaction_rate', 'conversion_rate', 'selectivity',
       'yield_pct', 'vibration_rms', 'motor_current', 'power_consumption_kw',
       'temp_setpoint', 'pressure_setpoint', 'efficiency_loss_pct']
        # copied and deleted the ones I don't need here #it makes no difference if the setpoints are calculated this way or with median, so I just calculate it like the rest

#and because I can, I wrote a function for that:
def away_with_nans(reactor, dataframe):
    mask = dataframe['reactor_id'] == reactor
    for i in columns:
        dataframe.loc[mask, i] = dataframe.loc[mask, i].interpolate(method='linear')

#ambient temp effect is now also calculated with the mean instead of forward fill

In [12]:
away_with_nans(2, df)
away_with_nans(3, df)

In [13]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 388800 entries, 0 to 388799
Data columns (total 20 columns):
 #   Column                Non-Null Count   Dtype         
---  ------                --------------   -----         
 0   timestamp             388800 non-null  datetime64[us]
 1   reactor_id            388800 non-null  int64         
 2   ambient_temp_effect   388799 non-null  float64       
 3   reactor_temp          388800 non-null  float64       
 4   reactor_pressure      388799 non-null  float64       
 5   feed_flow_rate        388800 non-null  float64       
 6   coolant_flow_rate     388800 non-null  float64       
 7   agitator_speed_rpm    388799 non-null  float64       
 8   reaction_rate         388800 non-null  float64       
 9   conversion_rate       388800 non-null  float64       
 10  selectivity           388800 non-null  float64       
 11  yield_pct             388800 non-null  float64       
 12  vibration_rms         388800 non-null  float64       
 13  motor_curr

In [14]:
#only time to fault left. Like I said before, I want to change it into a boolean column, since I have the time information already in the timestamp
df['normal_behavior'] = df['time_to_fault_min'].isna()

In [25]:
df[df['agitator_speed_rpm'].isna()]

,timestamp,reactor_id,ambient_temp_effect,reactor_temp,reactor_pressure,feed_flow_rate,coolant_flow_rate,agitator_speed_rpm,reaction_rate,conversion_rate,...,yield_pct,vibration_rms,motor_current,power_consumption_kw,temp_setpoint,pressure_setpoint,fault_type,efficiency_loss_pct,time_to_fault_min,normal_behavior
259200,2024-01-01,3,NaN,179.850553,NaN,99.608202,79.846183,NaN,0.719402,98.792799,...,81.646709,1.487815,45.46792,40.921128,180.0,12.0,0,0.0,NaN,True


In [27]:
df[df['timestamp'].dt.time == pd.Timestamp('00:00:00').time()] #-> only at midnight, the time is not written down (90 days x 3 reactors = 270 entries)

,timestamp,reactor_id,ambient_temp_effect,reactor_temp,reactor_pressure,feed_flow_rate,coolant_flow_rate,agitator_speed_rpm,reaction_rate,conversion_rate,...,yield_pct,vibration_rms,motor_current,power_consumption_kw,temp_setpoint,pressure_setpoint,fault_type,efficiency_loss_pct,time_to_fault_min,normal_behavior
0,2024-01-01,1,0.000000,181.135558,15.791013,101.108882,79.154645,305.779931,0.724542,99.151760,...,82.032893,1.470297,45.882315,41.294083,180.0,12.0,0,0.000000,NaN,True
1440,2024-01-02,1,0.697570,182.174831,15.599965,97.980473,79.532003,307.935879,0.728699,98.630419,...,81.434995,1.467984,46.124584,41.512125,180.0,12.0,0,0.000000,NaN,True
2880,2024-01-03,1,1.391742,182.557297,15.751081,99.023570,79.595328,306.817540,0.730229,99.300750,...,82.314387,1.518336,45.581746,41.023571,180.0,12.0,0,0.000000,NaN,True
4320,2024-01-04,1,2.079133,182.633464,15.639355,100.548078,81.216487,293.513762,0.730534,98.370423,...,81.172021,1.445671,45.749332,41.174399,180.0,12.0,0,0.000000,NaN,True
5760,2024-01-05,1,2.756394,182.956461,15.650508,99.673868,79.107901,291.711712,0.731826,98.789371,...,81.284519,1.488053,46.801095,42.120986,180.0,12.0,0,0.000000,NaN,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
381600,2024-03-26,3,-3.419771,183.580686,15.844313,98.846135,74.020136,303.840238,0.725997,98.985542,...,82.237064,1.480847,46.378471,41.740624,180.0,12.0,1,17.929293,385.0,False
383040,2024-03-27,3,-2.755928,179.340439,15.628276,100.019704,81.271018,297.532780,0.717362,99.177891,...,82.080060,1.488372,45.277503,40.749752,180.0,12.0,0,0.000000,NaN,True
384480,2024-03-28,3,-2.078658,180.146228,15.818193,101.329658,82.171735,298.592192,0.720585,98.948385,...,81.859775,1.497173,45.951215,41.356094,180.0,12.0,0,0.000000,NaN,True
385920,2024-03-29,3,-1.391262,180.026376,15.867198,99.361391,79.499838,302.656182,0.720106,99.095297,...,81.950100,1.451385,46.440992,41.796893,180.0,12.0,0,0.000000,NaN,True


In [28]:
df[df['reactor_id']==3]

,timestamp,reactor_id,ambient_temp_effect,reactor_temp,reactor_pressure,feed_flow_rate,coolant_flow_rate,agitator_speed_rpm,reaction_rate,conversion_rate,...,yield_pct,vibration_rms,motor_current,power_consumption_kw,temp_setpoint,pressure_setpoint,fault_type,efficiency_loss_pct,time_to_fault_min,normal_behavior
259200,2024-01-01 00:00:00,3,NaN,179.850553,NaN,99.608202,79.846183,NaN,0.719402,98.792799,...,81.646709,1.487815,45.467920,40.921128,180.0,12.0,0,0.0,NaN,True
259201,2024-01-01 00:01:00,3,4.848174e-04,180.236922,15.738926,100.307067,79.215299,296.580743,0.720948,98.628310,...,81.615648,1.481393,46.440511,41.796460,180.0,12.0,0,0.0,NaN,True
259202,2024-01-01 00:02:00,3,9.696348e-04,180.002165,15.712389,101.351489,79.356824,299.084656,0.720009,98.756093,...,81.235665,1.500604,45.998555,41.398700,180.0,12.0,0,0.0,NaN,True
259203,2024-01-01 00:03:00,3,1.454452e-03,180.132899,15.522936,99.714169,79.349225,301.589023,0.719516,98.569352,...,81.468642,1.563240,45.171085,40.653977,180.0,12.0,0,0.0,NaN,True
259204,2024-01-01 00:04:00,3,1.939270e-03,180.263633,15.570346,100.863559,81.842563,292.617797,0.721055,98.732429,...,81.020196,1.510412,45.554600,40.999140,180.0,12.0,0,0.0,NaN,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
388795,2024-03-30 23:55:00,3,-1.939270e-03,180.512489,15.698175,98.294404,79.056576,294.996530,0.722050,98.628870,...,81.132708,1.482459,45.609944,42.239260,180.0,12.0,0,0.0,NaN,True
388796,2024-03-30 23:56:00,3,-1.454452e-03,180.338679,15.717705,98.184695,82.652110,299.038660,0.721355,99.004749,...,81.402088,1.507740,45.391636,40.852472,180.0,12.0,0,0.0,NaN,True
388797,2024-03-30 23:57:00,3,-9.696348e-04,179.600059,15.700481,99.075055,80.572876,306.286187,0.718400,98.894388,...,81.457731,1.533022,46.423420,41.781078,180.0,12.0,0,0.0,NaN,True
388798,2024-03-30 23:58:00,3,-4.848174e-04,180.023226,15.650709,100.313890,81.067978,296.455683,0.720093,98.566737,...,81.293929,1.447584,45.742744,41.168469,180.0,12.0,0,0.0,NaN,True


In [29]:
#the first one is NaN, because it can't interpolate. So we have to adapt the function:
def away_with_nans(reactor, dataframe):
    mask = dataframe['reactor_id'] == reactor
    for i in columns:
        dataframe.loc[mask, i] = dataframe.loc[mask, i].interpolate(method='linear')
        dataframe.loc[mask, i] = dataframe.loc[mask, i].ffill()
        dataframe.loc[mask, i] = dataframe.loc[mask, i].bfill()

#like this, the function will first fill out all the NaNs with the interpolate method. If there are NaNs at the very end of the line, it will be filled out with ffill (the last valid entry fills out the next). After that, the remaining NaNs at the Beginning will be filled out with bfill (the next valid entry fills out the previous). Hence, all MV are gone. Yay!

In [30]:
away_with_nans(1, df)
away_with_nans(2, df)
away_with_nans(3, df)

In [31]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 388800 entries, 0 to 388799
Data columns (total 21 columns):
 #   Column                Non-Null Count   Dtype         
---  ------                --------------   -----         
 0   timestamp             388800 non-null  datetime64[us]
 1   reactor_id            388800 non-null  int64         
 2   ambient_temp_effect   388800 non-null  float64       
 3   reactor_temp          388800 non-null  float64       
 4   reactor_pressure      388800 non-null  float64       
 5   feed_flow_rate        388800 non-null  float64       
 6   coolant_flow_rate     388800 non-null  float64       
 7   agitator_speed_rpm    388800 non-null  float64       
 8   reaction_rate         388800 non-null  float64       
 9   conversion_rate       388800 non-null  float64       
 10  selectivity           388800 non-null  float64       
 11  yield_pct             388800 non-null  float64       
 12  vibration_rms         388800 non-null  float64       
 13  motor_curr

# Preprocessing for the stressed condition
Regime B — Stress Operation
- Higher temperature and pressure
- Reduced cooling efficiency
- Increased sensor noise
- Higher fault probability
- Represents harsh or high-throughput conditions

same thing with the stressed data

In [16]:
df_stressed = data[data['operating_regime']=='B'] #filtering for the condition
df_stressed['operating_regime'].value_counts()

operating_regime
B    388800
Name: count, dtype: int64

In [17]:
df_stressed.drop(columns='operating_regime', inplace=True)  #deleting the column
df_stressed.info()

<class 'pandas.DataFrame'>
RangeIndex: 388800 entries, 388800 to 777599
Data columns (total 20 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   timestamp             388800 non-null  str    
 1   reactor_id            388800 non-null  str    
 2   ambient_temp_effect   365439 non-null  float64
 3   reactor_temp          365350 non-null  float64
 4   reactor_pressure      365061 non-null  float64
 5   feed_flow_rate        365486 non-null  float64
 6   coolant_flow_rate     365519 non-null  float64
 7   agitator_speed_rpm    365509 non-null  float64
 8   reaction_rate         365486 non-null  float64
 9   conversion_rate       365528 non-null  float64
 10  selectivity           365306 non-null  float64
 11  yield_pct             365547 non-null  float64
 12  vibration_rms         365464 non-null  float64
 13  motor_current         365511 non-null  float64
 14  power_consumption_kw  365480 non-null  float64
 15  temp_s

In [18]:
df_stressed['timestamp']= pd.to_datetime(df_stressed['timestamp'])
df_stressed['reactor_id'] = df_stressed['reactor_id'].str[-1:] #so I only have the reactornumber as id
df_stressed['reactor_id'] = df_stressed['reactor_id'].astype('int')

In [19]:
df_stressed['reactor_id'].value_counts()

reactor_id
1    129600
2    129600
3    129600
Name: count, dtype: int64

In [32]:
away_with_nans(1, df_stressed)
away_with_nans(2, df_stressed)
away_with_nans(3, df_stressed)

In [33]:
df_stressed['normal_behavior'] = df_stressed['time_to_fault_min'].isna()
df_stressed.info()

<class 'pandas.DataFrame'>
RangeIndex: 388800 entries, 388800 to 777599
Data columns (total 21 columns):
 #   Column                Non-Null Count   Dtype         
---  ------                --------------   -----         
 0   timestamp             388800 non-null  datetime64[us]
 1   reactor_id            388800 non-null  int64         
 2   ambient_temp_effect   388800 non-null  float64       
 3   reactor_temp          388800 non-null  float64       
 4   reactor_pressure      388800 non-null  float64       
 5   feed_flow_rate        388800 non-null  float64       
 6   coolant_flow_rate     388800 non-null  float64       
 7   agitator_speed_rpm    388800 non-null  float64       
 8   reaction_rate         388800 non-null  float64       
 9   conversion_rate       388800 non-null  float64       
 10  selectivity           388800 non-null  float64       
 11  yield_pct             388800 non-null  float64       
 12  vibration_rms         388800 non-null  float64       
 13  motor

# exporting the data

In [34]:
df.to_csv('archive/baseline_cleaned.csv', index=False)
df_stressed.to_csv('archive/stressed_cleaned.csv', index=False)